# Knowledge Base Vectorization and Visualization

Build a searchable vector representation of the Insurellm knowledge base using LangChain, Hugging Face embeddings, and Chroma. The notebook loads Markdown documents, splits them into overlapping chunks, stores the embeddings, and visualizes the resulting vectors.

## Workflow

1. Load Markdown files and inspect the corpus.
2. Split documents into chunks.
3. Generate embeddings and persist them in Chroma.
4. Inspect the vector store and visualize embeddings with t-SNE.

## Requirements

Install the dependencies in your Python environment before running the notebook:

```bash
pip install langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface chromadb sentence-transformers scikit-learn plotly tiktoken
```

Place the source documents under `knowledge-base/`, organized into subdirectories such as `products/`, `employees/`, `contracts/`, and `company/`.

## 1. Imports and configuration

In [ ]:
from pathlib import Path

import glob
import numpy as np
import tiktoken
import plotly.graph_objects as go

from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE

KNOWLEDGE_BASE_DIR = Path("knowledge-base")
VECTOR_STORE_DIR = "vector_db"
COLLECTION_NAME = "insurellm_knowledge_base"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 1_000
CHUNK_OVERLAP = 200

## 2. Load the knowledge base

In [ ]:
markdown_files = sorted(KNOWLEDGE_BASE_DIR.glob("**/*.md"))

if not markdown_files:
    raise FileNotFoundError(
        f"No Markdown files found under '{KNOWLEDGE_BASE_DIR}'. "
        "Add your knowledge-base documents and rerun this section."
    )

corpus = "\n\n".join(
    file_path.read_text(encoding="utf-8")
    for file_path in markdown_files
)

print(f"Files found: {len(markdown_files):,}")
print(f"Total characters: {len(corpus):,}")

In [ ]:
encoding = tiktoken.get_encoding("o200k_base")
token_count = len(encoding.encode(corpus))

print(f"Approximate token count: {token_count:,}")

In [ ]:
documents = []

for folder in sorted(path for path in KNOWLEDGE_BASE_DIR.iterdir() if path.is_dir()):
    loader = DirectoryLoader(
        str(folder),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )

    folder_documents = loader.load()

    for document in folder_documents:
        document.metadata["doc_type"] = folder.name
        documents.append(document)

if not documents:
    raise ValueError("The knowledge base contains no loadable Markdown documents.")

print(f"Documents loaded: {len(documents):,}")

## 3. Split documents into chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = text_splitter.split_documents(documents)

if not chunks:
    raise ValueError("No chunks were generated from the loaded documents.")

print(f"Chunks created: {len(chunks):,}")
print(f"Average chunk length: {np.mean([len(chunk.page_content) for chunk in chunks]):.0f} characters")

## 4. Generate embeddings and build the vector store

The `all-MiniLM-L6-v2` model generates local sentence embeddings. Chroma persists the vectors in `vector_db/`.

**Note:** Rerunning this section replaces the existing collection with a fresh build from the current knowledge base.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

if Path(VECTOR_STORE_DIR).exists():
    existing_store = Chroma(
        collection_name=COLLECTION_NAME,
        persist_directory=VECTOR_STORE_DIR,
        embedding_function=embeddings,
    )
    existing_store.delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=VECTOR_STORE_DIR,
)

collection = vectorstore._collection
print(f"Vector store created with {collection.count():,} chunks.")

## 5. Inspect the vector store

In [ ]:
vector_count = collection.count()

if vector_count == 0:
    raise ValueError("The vector store is empty. Check the document-loading and embedding steps.")

sample = collection.get(limit=1, include=["embeddings"])
embedding_dimensions = len(sample["embeddings"][0])

print(f"Vectors: {vector_count:,}")
print(f"Embedding dimensions: {embedding_dimensions:,}")

## 6. Prepare vectors for visualization

In [ ]:
result = collection.get(include=["embeddings", "documents", "metadatas"])

vectors = np.asarray(result["embeddings"])
chunk_texts = result["documents"]
metadata = result["metadatas"]
doc_types = [item.get("doc_type", "other") for item in metadata]

type_colors = {
    "products": "blue",
    "employees": "green",
    "contracts": "red",
    "company": "orange",
}
colors = [type_colors.get(doc_type, "gray") for doc_type in doc_types]

print(f"Loaded {len(vectors):,} vectors for visualization.")

## 7. Visualize embeddings in 2D

In [ ]:
if len(vectors) < 2:
    raise ValueError("At least two vectors are required for t-SNE visualization.")

perplexity_2d = min(30, len(vectors) - 1)

vectors_2d = TSNE(
    n_components=2,
    perplexity=perplexity_2d,
    random_state=42,
    init="pca",
    learning_rate="auto",
).fit_transform(vectors)

hover_text = [
    f"Type: {doc_type}<br>Text: {content[:200]}..."
    for doc_type, content in zip(doc_types, chunk_texts)
]

fig_2d = go.Figure(
    data=[
        go.Scatter(
            x=vectors_2d[:, 0],
            y=vectors_2d[:, 1],
            mode="markers",
            marker={"size": 6, "color": colors, "opacity": 0.8},
            text=hover_text,
            hoverinfo="text",
        )
    ]
)

fig_2d.update_layout(
    title="Knowledge Base Embeddings — 2D t-SNE",
    xaxis_title="Component 1",
    yaxis_title="Component 2",
    template="plotly_white",
    width=900,
    height=650,
)

fig_2d.show()

## 8. Visualize embeddings in 3D

In [ ]:
if len(vectors) < 4:
    raise ValueError("At least four vectors are required for the 3D t-SNE visualization.")

perplexity_3d = min(30, len(vectors) - 1)

vectors_3d = TSNE(
    n_components=3,
    perplexity=perplexity_3d,
    random_state=42,
    init="pca",
    learning_rate="auto",
).fit_transform(vectors)

fig_3d = go.Figure(
    data=[
        go.Scatter3d(
            x=vectors_3d[:, 0],
            y=vectors_3d[:, 1],
            z=vectors_3d[:, 2],
            mode="markers",
            marker={"size": 4, "color": colors, "opacity": 0.8},
            text=hover_text,
            hoverinfo="text",
        )
    ]
)

fig_3d.update_layout(
    title="Knowledge Base Embeddings — 3D t-SNE",
    scene={
        "xaxis_title": "Component 1",
        "yaxis_title": "Component 2",
        "zaxis_title": "Component 3",
    },
    template="plotly_white",
    width=950,
    height=750,
)

fig_3d.show()